<a href="https://colab.research.google.com/github/marcory-hub/hailo-colab/blob/main/poDatabaseSort.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Mount google drive

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 2. Copy and unzip images from Google Drive
# Copy zip file to content directory
!cp /content/drive/MyDrive/poDatabase/media_4.zip /content/

# 3. Unzip images to images directory
!unzip /content/media_4.zip -d /content/

# Optional: Remove zip file after extraction
# !rm /content/poDatabaseImagesTest.zip

print("Images successfully copied and unzipped to /content/")


In [ ]:
# Install OpenAI CLIP library
!pip install git+https://github.com/openai/CLIP.git

post stamp

In [ ]:
# Libraries for image processing and zero-shot classification
import os
import torch
import clip
import csv
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Set up device (Colab T4 GPU)
device = "cuda"

# Load CLIP model
model, preprocess = clip.load("ViT-B/32", device=device)

# Define image categories with extremely detailed descriptions
categories = [
    "light microscopy (LM) with colors like yellow, pink and purple, white or light pastel or light grey background",

    "scanning electron microscopy (SEM) with high contrast grayscale images, extremely detailed surface topography, 3D-like depth perception, rough textured surfaces, showing intricate pollen grain external morphology, surface sculptures, pores, grey scale images",

    "transmission electron microscopy (TEM) flat 2D view, high contrast variations of different shades of gray, transparent sections showing cellular ultrastructure, detailed membrane systems, organelle details that took like circles, no surface texture",

    "flowering plants (FP) with natural green colors, multiple color ranges, outdoor garden or natural landscape setting, entire plant structures including stems, leaves, and flowers, colorful botanical scene with complex vegetation background, realistic plant photography",

    "line art (LA) black lines drawing, clean vector-like drawing, back dots and lines may be present, no gray uniform areas"
]

# Prepare text descriptions for zero-shot classification
text_inputs = torch.cat([clip.tokenize(f"a photo of {category}" for category in categories)]).to(device)

# Prepare CSV output
output_csv_path = '/content/image_type_classification.csv'
csv_output = []

# Process images
image_dir = '/content/media_1/'
for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_path = os.path.join(image_dir, filename)

        try:
            # Read image
            img = Image.open(image_path)

            # Determine display method based on image type
            display_img = img
            display_cmap = None

            # Check if it's a grayscale or electron microscopy image
            if len(np.array(img).shape) == 2 or (len(np.array(img).shape) > 2 and np.array(img).shape[2] == 1):
                display_cmap = 'gray'

            # Load and preprocess the image
            image = preprocess(img).unsqueeze(0).to(device)

            # Get image features
            with torch.no_grad():
                image_features = model.encode_image(image)
                text_features = model.encode_text(text_inputs)

            # Compute similarity
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            probabilities = similarity.cpu().numpy()[0]

            # Get the most likely category
            max_prob_index = probabilities.argmax()
            predicted_type = categories[max_prob_index].split('(')[1].split(')')[0]

            # Create a figure with postage stamp style image
            plt.figure(figsize=(2, 2))

            # Display image with appropriate color handling
            if display_cmap:
                plt.imshow(display_img, cmap=display_cmap)
            else:
                plt.imshow(display_img)

            plt.title(f"{filename}\nPredicted: {predicted_type}")
            plt.axis('off')
            plt.tight_layout()
            plt.show()

            # Store result
            csv_output.append([filename, predicted_type])

            print(f"Image: {filename}")
            for cat, prob in zip(categories, probabilities):
                print(f"{cat.split('(')[0].strip()}: {prob:.2f}")
            print("\n")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# Write results to CSV
with open(output_csv_path, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow(['Filename', 'ImageType'])  # Header
    csvwriter.writerows(csv_output)

print(f"Classification results saved to {output_csv_path}")

In [ ]:
# batch processing

In [ ]:
# Libraries for image processing and zero-shot classification
import os
import torch
import clip
import csv
from PIL import Image
import numpy as np
import logging

# Set up logging
logging.basicConfig(filename='/content/image_classification_log.txt', level=logging.INFO)

# Set up device (Colab T4 GPU)
device = "cuda"

# Load CLIP model
model, preprocess = clip.load("ViT-B/32", device=device)

# Define image categories with extremely detailed descriptions
categories = [
    "light microscopy (LM) with colors like yellow, pink and purple, white or light pastel or light grey background",

    "scanning electron microscopy (SEM) with high contrast grayscale images, extremely detailed surface topography, 3D-like depth perception, rough textured surfaces, showing intricate pollen grain external morphology, surface sculptures, pores, grey scale images",

    "transmission electron microscopy (TEM) flat 2D view, high contrast variations of different shades of gray, transparent sections showing cellular ultrastructure, detailed membrane systems, organelle details that took like circles, no surface texture",

    "flowering plants (FP) with natural green colors, multiple color ranges, outdoor garden or natural landscape setting, entire plant structures including stems, leaves, and flowers, colorful botanical scene with complex vegetation background, realistic plant photography",

    "line art (LA) black lines drawing, clean vector-like drawing, back dots and lines may be present, no gray uniform areas"
]

# Prepare text descriptions for zero-shot classification
text_inputs = torch.cat([clip.tokenize(f"a photo of {category}" for category in categories)]).to(device)

# Prepare CSV output
output_csv_path = '/content/image_type_classification.csv'
csv_output = []

# Process images
image_dir = '/content/media_4/'
for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        image_path = os.path.join(image_dir, filename)

        try:
            # Read image
            img = Image.open(image_path)

            # Load and preprocess the image
            image = preprocess(img).unsqueeze(0).to(device)

            # Get image features
            with torch.no_grad():
                image_features = model.encode_image(image)
                text_features = model.encode_text(text_inputs)

            # Compute similarity
            similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
            probabilities = similarity.cpu().numpy()[0]

            # Get the most likely category
            max_prob_index = probabilities.argmax()
            predicted_type = categories[max_prob_index].split('(')[1].split(')')[0]

            # Store result
            csv_output.append([filename, predicted_type])

            # Log the classification
            logging.info(f"Image: {filename}, Predicted Type: {predicted_type}")

        except Exception as e:
            logging.error(f"Error processing {filename}: {e}")
            continue

# Write results to CSV
with open(output_csv_path, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    csvwriter.writerow(['Filename', 'ImageType'])  # Header
    csvwriter.writerows(csv_output)

logging.info(f"Classification results saved to {output_csv_path}")
print(f"Classification results saved to {output_csv_path}")

In [ ]:
!rm -rf /content/media_3